# Model 2 — COCO-10 Benchmark

### Grounding DINO Tiny → EfficientSAM-S ONNX → LaMa ONNX

This notebook runs the **same end-to-end Model 2 pipeline** on a fixed, reproducible set of 10 COCO `val2017` images. It records grounding, EfficientSAM encoder/decoder, segmentation, MI-GAN, and end-to-end CPU latency, while saving the grounding box, segmentation mask, and final object-removal result for visual inspection.

The first run creates `benchmark_data/coco10_manifest.json`. Reuse that manifest for Model 1 and Model 2 to keep the image/prompt set identical across comparisons.

**Expected project layout:** place this notebook inside `Model2_GroundingDINO_EfficientSAM_S_LaMa/`, next to `pipeline.py` and `model_wts/`.

Sources: COCO `val2017` contains 5,000 validation images with instance segmentation annotations; EfficientSAM provides a split ONNX encoder/decoder and supports box prompting. citeturn314069search0turn314069search7turn203446search0

In [ ]:
# 0. Imports + paths
from pathlib import Path
import json, os, sys, time, math, zipfile, urllib.request
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "pipeline.py").exists():
    # Allow running from the parent directory.
    candidate = ROOT / "Model2_GroundingDINO_EfficientSAM_S_LaMa"
    if (candidate / "pipeline.py").exists():
        ROOT = candidate

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "benchmark_data"
IMG_DIR = DATA_DIR / "coco10_images"
OUT_DIR = ROOT / "benchmark_outputs"
DATA_DIR.mkdir(exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("Weights:", ROOT / "model_wts")
print("Outputs:", OUT_DIR)

## 1. Benchmark configuration

`REUSE_MANIFEST=True` means the notebook will reuse the previously created 10-image/prompt manifest. Set it to `False` only when you intentionally want to regenerate the sample.

The sample is deterministic: ten common COCO object categories are selected from `val2017` annotations. Only those ten image files are downloaded; the large COCO validation archive is not required.

In [ ]:
REUSE_MANIFEST = True
MANIFEST_PATH = DATA_DIR / "coco10_manifest.json"

# Keep these identical when comparing Model 1 vs Model 2.
TARGET_CATEGORIES = [
    "person", "bicycle", "car", "motorcycle", "bus",
    "dog", "horse", "cat", "bottle", "chair"
]

# CPU benchmark settings
GROUNDING_BOX_THRESHOLD = 0.30
GROUNDING_TEXT_THRESHOLD = 0.25
BOX_EXPAND_RATIO = 0.03
CLOSING_KERNEL = 3
DILATION_KERNEL = 3
WARMUP = True
SAVE_OUTPUTS = True

In [ ]:
# 2. COCO utilities: download annotations once, pick deterministic image/category pairs,
#    then download only the 10 required images.

COCO_ANN_ZIP = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
COCO_IMG_BASE = "http://images.cocodataset.org/val2017/"
ANN_ZIP_PATH = DATA_DIR / "annotations_trainval2017.zip"
ANN_JSON_PATH = DATA_DIR / "instances_val2017.json"


def download(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    print(f"Downloading: {url}")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as r, open(dest, "wb") as f:
        total = int(r.headers.get("Content-Length", "0"))
        done = 0
        while True:
            chunk = r.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)
            done += len(chunk)
            if total:
                print(f"\r{done/total*100:5.1f}%", end="")
    print()
    return dest


def ensure_instances_json():
    if ANN_JSON_PATH.exists() and ANN_JSON_PATH.stat().st_size > 0:
        return ANN_JSON_PATH
    download(COCO_ANN_ZIP, ANN_ZIP_PATH)
    with zipfile.ZipFile(ANN_ZIP_PATH) as z:
        member = "annotations/instances_val2017.json"
        if member not in z.namelist():
            raise FileNotFoundError(f"{member} not found in {ANN_ZIP_PATH}")
        with z.open(member) as src, open(ANN_JSON_PATH, "wb") as dst:
            shutil.copyfileobj(src, dst)
    return ANN_JSON_PATH


def build_manifest():
    ann_path = ensure_instances_json()
    with open(ann_path, "r") as f:
        coco = json.load(f)

    images = {x["id"]: x for x in coco["images"]}
    categories = {x["id"]: x["name"] for x in coco["categories"]}
    cat_to_id = {v: k for k, v in categories.items()}

    anns_by_cat = {name: [] for name in TARGET_CATEGORIES}
    for a in coco["annotations"]:
        name = categories.get(a["category_id"])
        if name in anns_by_cat and not a.get("iscrowd", 0):
            x, y, w, h = a["bbox"]
            img = images[a["image_id"]]
            area_ratio = (w * h) / max(1.0, img["width"] * img["height"])
            # Prefer visible, reasonably sized objects. Fall back later if needed.
            anns_by_cat[name].append((area_ratio, a))

    manifest = []
    for category in TARGET_CATEGORIES:
        candidates = anns_by_cat.get(category, [])
        if not candidates:
            raise RuntimeError(f"No COCO val2017 candidate found for category: {category}")
        # Deterministic selection: medium-sized object, then image id / annotation id.
        candidates = sorted(candidates, key=lambda t: (abs(t[0] - 0.08), images[t[1]["image_id"]]["id"], t[1]["id"]))
        _, ann = candidates[0]
        img = images[ann["image_id"]]
        manifest.append({
            "image_id": img["id"],
            "file_name": img["file_name"],
            "width": img["width"],
            "height": img["height"],
            "prompt": category,
            "target_category": category,
            "target_annotation_id": ann["id"],
            "target_bbox_xywh": ann["bbox"],
        })

    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2)
    return manifest

if REUSE_MANIFEST and MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text())
else:
    manifest = build_manifest()

print(pd.DataFrame(manifest)[["image_id", "file_name", "prompt"]].to_string(index=False))

In [ ]:
# 3. Download the 10 COCO images only.
for item in manifest:
    path = IMG_DIR / item["file_name"]
    if not path.exists() or path.stat().st_size == 0:
        download(COCO_IMG_BASE + item["file_name"], path)

# Verify dimensions and show the ten inputs.
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, item in zip(axes.ravel(), manifest):
    img = Image.open(IMG_DIR / item["file_name"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(f'{item["image_id"]}: {item["prompt"]}')
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Load the local Model 2 pipeline

This notebook does not download model weights at inference time. `pipeline.py` loads Grounding DINO from `model_wts/grounding_dino`, EfficientSAM-S encoder/decoder from `model_wts/efficient_sam`, and LaMa from `model_wts/lama`.

In [ ]:
from pipeline import Model2Pipeline, Config

cfg = Config(
    grounding_box_threshold=GROUNDING_BOX_THRESHOLD,
    grounding_text_threshold=GROUNDING_TEXT_THRESHOLD,
    box_expand_ratio=BOX_EXPAND_RATIO,
    closing_kernel=CLOSING_KERNEL,
    dilation_kernel=DILATION_KERNEL,
    max_detections=5,
    intra_op_threads=0,
    inter_op_threads=0,
)

pipeline = Model2Pipeline(cfg)
print("Model 2 loaded successfully on CPU.")

In [ ]:
# 5. Optional warm-up outside the measured 10-image results.
if WARMUP:
    warm_item = manifest[0]
    warm_img = Image.open(IMG_DIR / warm_item["file_name"]).convert("RGB")
    print("Running warm-up on", warm_item["file_name"], "prompt=", warm_item["prompt"])
    _ = pipeline.remove(warm_img, warm_item["prompt"])
    print("Warm-up complete.")

In [ ]:
# 6. Run the complete benchmark.
rows = []
results = []

for i, item in enumerate(manifest, start=1):
    img_path = IMG_DIR / item["file_name"]
    image = Image.open(img_path).convert("RGB")
    print(f"[{i}/10] {item['file_name']} | prompt={item['prompt']}")

    t0 = time.perf_counter()
    try:
        result = pipeline.remove(image, item["prompt"])
        wall_total_ms = (time.perf_counter() - t0) * 1000.0
        result["wall_total_ms"] = wall_total_ms
        results.append((item, result))

        lat = result["latency"]
        g = result["grounding"]
        s = result["segmentation"]
        rows.append({
            "image_id": item["image_id"],
            "file_name": item["file_name"],
            "prompt": item["prompt"],
            "grounding_score": g["best_score"],
            "grounding_ms": lat["grounding_ms"],
            "efficient_sam_encoder_ms": lat["efficient_sam_encoder_ms"],
            "efficient_sam_decoder_ms": lat["efficient_sam_decoder_ms"],
            "segmentation_total_ms": lat["segmentation_total_ms"],
            "lama_ms": lat["lama_ms"],
            "pipeline_total_ms": lat["total_ms"],
            "wall_total_ms": wall_total_ms,
            "mask_area_px": int((result["mask"] > 0).sum()),
            "mask_area_ratio": float((result["mask"] > 0).mean()),
            "status": "ok",
            "error": "",
        })
    except Exception as e:
        print("  ERROR:", repr(e))
        rows.append({
            "image_id": item["image_id"],
            "file_name": item["file_name"],
            "prompt": item["prompt"],
            "grounding_score": np.nan,
            "grounding_ms": np.nan,
            "efficient_sam_encoder_ms": np.nan,
            "efficient_sam_decoder_ms": np.nan,
            "segmentation_total_ms": np.nan,
            "lama_ms": np.nan,
            "pipeline_total_ms": np.nan,
            "wall_total_ms": np.nan,
            "mask_area_px": np.nan,
            "mask_area_ratio": np.nan,
            "status": "error",
            "error": repr(e),
        })

bench = pd.DataFrame(rows)
bench.to_csv(OUT_DIR / "coco10_benchmark_results.csv", index=False)
print("\nSaved:", OUT_DIR / "coco10_benchmark_results.csv")
bench

## 7. Visual inspection: box → mask → final removal

Each example is saved individually as a four-panel image so that segmentation mistakes can be diagnosed separately from grounding mistakes.

In [ ]:
def save_visual(item, result):
    original = result["original"]
    box_img = result["grounding_overlay"]
    mask_img = result["mask_overlay"]
    final = result["result"]
    box = result["box"]
    score = result["grounding"]["best_score"]
    seg_score = result["segmentation"]["score"]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    panels = [original, box_img, mask_img, final]
    titles = [
        f'Input\nprompt: {item["prompt"]}',
        f'Grounding DINO\nscore={score:.3f}\nbox={np.round(box).astype(int).tolist()}',
        f'EfficientSAM-S mask\nscore={seg_score:.3f}',
        f'LaMa result\n{result["latency"]["total_ms"]:.0f} ms total',
    ]
    for ax, panel, title in zip(axes, panels, titles):
        ax.imshow(panel)
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(f'COCO image {item["image_id"]} — {item["file_name"]}', fontsize=14)
    plt.tight_layout()
    out = OUT_DIR / f'{item["image_id"]}_{item["prompt"].replace(" ", "_")}.png'
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return out

for item, result in results:
    save_visual(item, result)

## 8. Latency summary

Use these values for the Model 1 vs Model 2 comparison. `pipeline_total_ms` is the pipeline's internal end-to-end timer; `wall_total_ms` includes the notebook/Python call overhead around the call.

In [ ]:
metric_cols = [
    "grounding_ms", "efficient_sam_encoder_ms", "efficient_sam_decoder_ms",
    "segmentation_total_ms", "lama_ms", "pipeline_total_ms", "wall_total_ms"
]
summary = bench[bench["status"] == "ok"][metric_cols].agg(["mean", "median", "min", "max"]).round(2)
summary

In [ ]:
# Bar chart of mean per-stage latency.
ok = bench[bench["status"] == "ok"]
if len(ok):
    means = ok[[
        "grounding_ms", "efficient_sam_encoder_ms", "efficient_sam_decoder_ms", "lama_ms"
    ]].mean()
    plt.figure(figsize=(11, 5))
    plt.bar(means.index, means.values)
    plt.ylabel("Mean CPU latency (ms)")
    plt.title("Model 2 — Mean Stage Latency on COCO-10")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

## 9. Optional COCO ground-truth overlap evaluation

Because each manifest row stores the target annotation's COCO bounding box, the notebook can measure whether Grounding DINO localized the requested category. It also reports the best box IoU against **any instance of the same category in that image**, which is appropriate for a category-level text prompt.

Mask IoU is enabled when `pycocotools` is installed. If it is not installed, the notebook still completes the full inference benchmark and leaves mask-IoU columns as `NaN`.

In [ ]:
# Build lightweight COCO maps for optional GT evaluation.
try:
    ensure_instances_json()
    coco_data = json.loads(ANN_JSON_PATH.read_text())
    image_by_id = {x["id"]: x for x in coco_data["images"]}
    cat_by_id = {x["id"]: x["name"] for x in coco_data["categories"]}
    anns_by_image = {}
    for a in coco_data["annotations"]:
        if not a.get("iscrowd", 0):
            anns_by_image.setdefault(a["image_id"], []).append(a)
    HAS_COCO_GT = True
except Exception as e:
    HAS_COCO_GT = False
    print("Could not load COCO GT annotations:", e)


def xywh_to_xyxy(b):
    x, y, w, h = [float(v) for v in b]
    return np.array([x, y, x + w, y + h], dtype=np.float32)


def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return float(inter / union) if union > 0 else 0.0

mask_utils = None
try:
    from pycocotools import mask as mask_utils
except Exception:
    print("pycocotools not installed; box IoU will still be measured, mask IoU will be skipped.")


def decode_ann_mask(ann, height, width):
    if mask_utils is None:
        return None
    seg = ann.get("segmentation")
    if isinstance(seg, list):
        rles = mask_utils.frPyObjects(seg, height, width)
        rle = mask_utils.merge(rles)
    elif isinstance(seg, dict):
        rle = seg
        if isinstance(rle.get("counts"), list):
            rle = mask_utils.frPyObjects(rle, height, width)
    else:
        return None
    return mask_utils.decode(rle).astype(bool)

if HAS_COCO_GT:
    gt_rows = []
    for item, result in results:
        img_id = item["image_id"]
        catsame = [a for a in anns_by_image.get(img_id, []) if cat_by_id.get(a["category_id"]) == item["target_category"]]
        pred_box = result["box"]
        box_ious = [box_iou(pred_box, xywh_to_xyxy(a["bbox"])) for a in catsame]
        gt_best_box_iou = max(box_ious) if box_ious else np.nan

        pred_mask = result["mask"] > 0
        mask_ious = []
        h, w = result["original"].shape[:2]
        for a in catsame:
            gm = decode_ann_mask(a, h, w)
            if gm is None:
                continue
            inter = np.logical_and(pred_mask, gm).sum()
            union = np.logical_or(pred_mask, gm).sum()
            mask_ious.append(float(inter / union) if union else 0.0)
        gt_best_mask_iou = max(mask_ious) if mask_ious else np.nan

        gt_rows.append({
            "image_id": img_id,
            "prompt": item["prompt"],
            "best_gt_box_iou": gt_best_box_iou,
            "best_gt_mask_iou": gt_best_mask_iou,
        })

    gt_df = pd.DataFrame(gt_rows)
    bench = bench.merge(gt_df, on=["image_id", "prompt"], how="left")
    bench.to_csv(OUT_DIR / "coco10_benchmark_results_with_gt.csv", index=False)
    display(bench[["image_id", "prompt", "grounding_score", "best_gt_box_iou", "best_gt_mask_iou"]])

## 10. Final benchmark report

The following table is the compact report to paste into the Model 1 vs Model 2 comparison. Keep the same `coco10_manifest.json`, prompts, and warm-up policy across both models.

In [ ]:
report_cols = [
    "image_id", "prompt", "grounding_score", "grounding_ms",
    "efficient_sam_encoder_ms", "efficient_sam_decoder_ms",
    "lama_ms", "pipeline_total_ms", "best_gt_box_iou", "best_gt_mask_iou", "status"
]
report = bench[[c for c in report_cols if c in bench.columns]].copy()
report.round(3)

In [ ]:
# Aggregate numbers for your report/slide.
ok = bench[bench["status"] == "ok"]
agg = {
    "n_success": int(len(ok)),
    "n_total": int(len(bench)),
    "mean_grounding_ms": float(ok["grounding_ms"].mean()) if len(ok) else np.nan,
    "mean_efficient_sam_encoder_ms": float(ok["efficient_sam_encoder_ms"].mean()) if len(ok) else np.nan,
    "mean_efficient_sam_decoder_ms": float(ok["efficient_sam_decoder_ms"].mean()) if len(ok) else np.nan,
    "mean_lama_ms": float(ok["lama_ms"].mean()) if len(ok) else np.nan,
    "mean_pipeline_total_ms": float(ok["pipeline_total_ms"].mean()) if len(ok) else np.nan,
    "median_pipeline_total_ms": float(ok["pipeline_total_ms"].median()) if len(ok) else np.nan,
    "mean_best_gt_box_iou": float(ok["best_gt_box_iou"].mean()) if "best_gt_box_iou" in ok else np.nan,
    "mean_best_gt_mask_iou": float(ok["best_gt_mask_iou"].mean()) if "best_gt_mask_iou" in ok else np.nan,
}
print(json.dumps(agg, indent=2))
Path(OUT_DIR / "coco10_summary.json").write_text(json.dumps(agg, indent=2))

### Reproducibility note

For an apples-to-apples Model 1 vs Model 2 comparison, copy `benchmark_data/coco10_manifest.json` from the first benchmark project into the second project and set `REUSE_MANIFEST=True`. This fixes the **same ten COCO images and the same text prompts** while allowing the segmentation/inpainting stack to change.

COCO `val2017` provides public instance segmentation annotations for validation images, and the EfficientSAM project documents a separate ONNX encoder/decoder deployment path with box prompting. citeturn314069search0turn203446search0turn203446search6